# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

import ibis as ib
from ibis import _

from iacs.architect import Architect

ib.options.interactive = True  # auto-display results (like pandas)

## Load components.yaml

In [ ]:
import atexit

COMPONENTS_DIR = Path.cwd().parent / "builtins"

a = Architect.from_manifest(COMPONENTS_DIR)
atexit.register(a.registry.close)

## Evaluate highest-priority tasks to work on

In [ ]:
entity_ids = a.get("entity_id")
entity_ids

In [ ]:
parents = a.get("parent")
parents

In [ ]:
reqs = a.get("requirement")
reqs

In [ ]:
# Highest priority leaves
(
    reqs
    .join(parents, reqs.entity_id == parents.parent_id, how="anti")
    .join(entity_ids.select(_.value, _.entity_key, _.path), _.entity_id == entity_ids.value )
    .order_by(ib.desc(_.priority))
    .select(_.entity_key, _.priority, _.value.name("requirement_type"))
)

In [ ]:
a.get("solution")

In [ ]:
a.load_dataflow("derive_components")

In [ ]:
a.execute("field_types_with_entity_ref", validated_registry=a.registry)